# Saathi: LoRA fine-tuning of Qwen3.5-4B (free Colab T4 or Kaggle T4)

**Goal:** teach the small model the *tone and language* of a warm peer supporter in English, Devanagari Hindi and Hinglish. Topic filtering and crisis handling are NOT learned here; the app's guardrails and fixed crisis templates do that.

**How to run (about 45-60 min on a free T4):**
- **Colab:** `Runtime -> Change runtime type -> T4 GPU`. Cell 2 asks you to upload `sft_train.jsonl` and `sft_val.jsonl` from `data/processed/`.
- **Kaggle:** `Settings -> Accelerator -> GPU T4`, `Internet -> on`. Add the two files as a dataset (`+ Add Input -> Upload`); cell 2 finds them under `/kaggle/input`. Outputs appear in the right panel under `/kaggle/working` (Output) for download.
- Run the cells top to bottom. **Cell 7 saves the adapter and GGUF straight after training**; the language test is last, so an error there can no longer lose the trained model.

**Run 1 (2 epochs) result:** validation loss 1.4443 after epoch 1, **1.4621 after epoch 2** (training loss fell from ~1.39 to ~1.03): epoch 2 overfits. This run therefore uses **1 epoch**, which also halves the GPU time.

**Deviation from the original brief (explain in the report):** the brief said 4-bit QLoRA. Unsloth's Qwen3.5 guide says 4-bit QLoRA is *not recommended* for Qwen3.5 because the quantisation error is unusually large, and recommends 16-bit LoRA (~10 GB VRAM for 4B, which fits the T4's 16 GB). Everything else follows the brief: r=16, alpha=32, dropout 0.05, all attention + MLP projections, LR 2e-4, max length 2048, loss on assistant tokens only, adapter saved separately; epochs 2 -> 1 based on run 1's validation loss.

In [ ]:
# 1. Install Unsloth (Colab already has torch + CUDA)
!pip install -q unsloth
import torch
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# 2. Upload the training data built by scripts/prepare_sft_data.py
import os, json, glob
from collections import Counter
ON_KAGGLE = os.path.exists("/kaggle/working")   # Colab also has a /kaggle folder
OUT = "/kaggle/working" if ON_KAGGLE else "/content"
os.makedirs(OUT, exist_ok=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name)
    return hits[0] if hits else None

if not (find("sft_train.jsonl") and find("sft_val.jsonl")):
    from google.colab import files
    files.upload()   # choose sft_train.jsonl and sft_val.jsonl

def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def download(path):
    # Colab: browser download. Kaggle: the file is already in /kaggle/working -> Output panel.
    print("saved:", path, f"({os.path.getsize(path) / 1e6:.0f} MB)")
    if not ON_KAGGLE:
        from google.colab import files
        files.download(path)

train_rows, val_rows = load(find("sft_train.jsonl")), load(find("sft_val.jsonl"))
print(len(train_rows), "train", Counter(r["lang"] for r in train_rows))
print(len(val_rows), "val", Counter(r["lang"] for r in val_rows))
print(json.dumps(train_rows[0]["messages"][1:], ensure_ascii=False, indent=1)[:600])

In [ ]:
# 3. Load Qwen3.5-4B in 16-bit and add LoRA adapters
from unsloth import FastLanguageModel
MAX_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-4B",
    max_seq_length=MAX_LEN,
    load_in_4bit=False,      # QLoRA not recommended for Qwen3.5 (Unsloth docs)
    load_in_16bit=True,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
# 4. Format with the model's own chat template (the same family of template Ollama uses)
from datasets import Dataset

def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}

train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
val_ds = Dataset.from_list(val_rows).map(to_text, remove_columns=["messages"])
lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:300]]
print("tokens per example (first 300): max", max(lengths), "mean", sum(lengths) // len(lengths))
print(train_ds[0]["text"][-700:])

In [ ]:
# 5. Trainer: loss on assistant tokens only, evaluate each epoch, keep the best checkpoint
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

use_bf16 = torch.cuda.is_bf16_supported()   # False on T4 -> fp16
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch size 8
        num_train_epochs=1,                 # run 1: eval loss rose in epoch 2 (1.444 -> 1.462)
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",                 # we save the adapter ourselves in cell 7
        fp16=not use_bf16,
        bf16=use_bf16,
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs",
        report_to="none",                   # set to "wandb" if you have an account
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
# Sanity check: only the assistant reply should be trained on (labels != -100)
ex = trainer.train_dataset[0]
kept = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("TRAINED ON:", tokenizer.decode(kept)[:400])

In [ ]:
# 6. Train, then write the full log to a file (download it with the adapter)
stats = trainer.train()
print(stats)
with open(f"{OUT}/finetune_log.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
for row in trainer.state.log_history:
    if "eval_loss" in row:
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

In [ ]:
# 7. SAVE FIRST: LoRA adapter (small) and training log, before anything that could fail
model.save_pretrained(f"{OUT}/saathi_lora")
tokenizer.save_pretrained(f"{OUT}/saathi_lora")
!cd {OUT} && zip -qr saathi_lora.zip saathi_lora finetune_log.json
download(f"{OUT}/saathi_lora.zip")

In [ ]:
# 8. Export a 4-bit GGUF for Ollama (merges the adapter; takes ~10-15 min)
model.save_pretrained_gguf(f"{OUT}/saathi_gguf", tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob(f"{OUT}/saathi_gguf*/*.gguf") + glob.glob(f"{OUT}/*.gguf") + glob.glob("*.gguf"))
print(gguf)
download([g for g in gguf if "q4_k_m" in g.lower()][-1] if any("q4_k_m" in g.lower() for g in gguf) else gguf[-1])
# On your laptop: put the .gguf in models/ and run
#   uv run python scripts/make_ollama_model.py models/<file>.gguf
# It creates the Ollama model `saathi-ft` with the same chat template as qwen3.5:4b.

In [ ]:
# 9. Optional quick check in all three languages (everything is already saved above)
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)   # Qwen3.5 loads a multimodal processor
persona = train_rows[0]["messages"][0]["content"].split("\n\nLanguage:")[0]
tests = [
    ("Reply in English.", "I moved to Bangalore for work and I just feel so alone here"),
    ("Reply in Hinglish using Roman script, the way the user writes. Mix English words in naturally, as they do.",
     "yaar ghar wale har baat pe cousin se compare karte hain, thak gaya hu"),
    ("Reply in Hindi using Devanagari script.",
     "ऑफिस का प्रेशर पागल कर देगा, बॉस रात ग्यारह बजे भी मैसेज करते हैं"),
]
for instr, user in tests:
    msgs = [{"role": "system", "content": f"{persona}\n\nLanguage: {instr}"},
            {"role": "user", "content": user}]
    try:
        prompt = text_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = text_tok(prompt, return_tensors="pt").input_ids.to("cuda")
        out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.6, do_sample=True)
        print("USER:", user, "\nSAATHI:", text_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), "\n")
    except Exception as e:
        print("test skipped:", type(e).__name__, e)   # not needed; the laptop eval covers this